In [38]:
import os
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

In [40]:
os.listdir('/content/drive/MyDrive/Dataset')

['WIN_20260207_10_59_32_Pro.jpg',
 'acne-open-comedo-59.jpeg',
 'train',
 'Dataset.zip',
 'testing',
 'facial_model.keras']

In [41]:
train_dataset = "/content/drive/MyDrive/Dataset/train"
test_dataset = "/content/drive/MyDrive/Dataset/testing"

In [42]:
os.listdir(train_dataset)

['Wrinkle', 'Rosacea', 'Normal', 'Acne', 'Hyperpigmentation', 'Eczemaa']

In [43]:
os.listdir(test_dataset)

['Acne', 'Eczemaa', 'Hyperpigmentation', 'Normal', 'Rosacea', 'Wrinkle']

In [44]:
train_count = {}
for image in os.listdir(train_dataset):
    image_path = os.path.join(train_dataset,image)
    train_count[image] = len(os.listdir(image_path))
print(train_count)

{'Wrinkle': 1000, 'Rosacea': 1086, 'Normal': 1142, 'Acne': 1092, 'Hyperpigmentation': 1096, 'Eczemaa': 1154}


In [45]:
test_count = {}
for image in os.listdir(test_dataset):
    image_path = os.path.join(test_dataset,image)
    test_count[image] = len(os.listdir(image_path))
print(test_count)

{'Acne': 222, 'Eczemaa': 246, 'Hyperpigmentation': 117, 'Normal': 346, 'Rosacea': 145, 'Wrinkle': 100}


In [46]:
classes = test_count.keys()
print(classes)

dict_keys(['Acne', 'Eczemaa', 'Hyperpigmentation', 'Normal', 'Rosacea', 'Wrinkle'])


In [47]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

In [48]:
train_data = image_dataset_from_directory(
    train_dataset,
    validation_split = 0.2,
    subset = "training",
    color_mode = "rgb",
    image_size = (224,224),
    shuffle = True,
    seed = 42,
    batch_size = 32
    )

val_data = image_dataset_from_directory(
    train_dataset,
    validation_split = 0.2,
    subset = "validation",
    color_mode = "rgb",
    image_size =(224,224),
    shuffle = False,
    seed = 42,
    batch_size = 32
)

Found 6570 files belonging to 6 classes.
Using 5256 files for training.
Found 6570 files belonging to 6 classes.
Using 1314 files for validation.


In [49]:
test_data = image_dataset_from_directory(
    test_dataset,
    color_mode = "rgb",
    image_size = (224,224),
    shuffle = False,
    
    seed = 42,
    batch_size = 32
    )

Found 1176 files belonging to 6 classes.


In [50]:
from tensorflow.keras.applications.efficientnet import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

val_data = val_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

test_data = test_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

train_data = train_data.prefetch(AUTOTUNE)
val_data = val_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)

In [51]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential


In [52]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)


base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

In [53]:
base_model.trainable = False

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1)
])

In [54]:
model = Sequential([
    data_augmentation,
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(6, activation="softmax")
])

: 

: 

In [55]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [56]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2
    )
]

In [76]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=40,
    callbacks=callbacks
)

Epoch 1/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 89ms/step - accuracy: 0.8896 - loss: 0.3473 - val_accuracy: 0.9216 - val_loss: 0.2525 - learning_rate: 1.6000e-07
Epoch 2/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - accuracy: 0.8858 - loss: 0.3554 - val_accuracy: 0.9216 - val_loss: 0.2524 - learning_rate: 1.6000e-07
Epoch 3/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 93ms/step - accuracy: 0.8927 - loss: 0.3407 - val_accuracy: 0.9216 - val_loss: 0.2524 - learning_rate: 3.2000e-08
Epoch 4/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - accuracy: 0.8900 - loss: 0.3414 - val_accuracy: 0.9216 - val_loss: 0.2524 - learning_rate: 3.2000e-08
Epoch 5/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step - accuracy: 0.8893 - loss: 0.3406 - val_accuracy: 0.9216 - val_loss: 0.2524 - learning_rate: 6.4000e-09
Epoch 6/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - accuracy: 0.8898 - loss: 0.3425 - val_accuracy: 0.9216 - val_loss: 0.2524 - learning_rate: 6.4000e-09
Epoch 7/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/ste

In [77]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,543,799 (17.33 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 329,486 (1.26 MB)

In [78]:
model.evaluate(test_data)

37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.7313 - loss: 0.7363


[0.7363321185112, 0.7312925457954407]

In [60]:
model.save("/content/drive/MyDrive/Dataset/facial_model.keras")

In [61]:
from PIL import Image

In [62]:
image = "/content/drive/MyDrive/Dataset/acne-open-comedo-59.jpeg"

In [63]:
import numpy as np

In [64]:
from tensorflow.keras.applications.efficientnet import preprocess_input

img = Image.open(image)
img = img.convert("RGB")
img = img.resize((224,224))

img = np.array(img).astype(np.float32)

img = preprocess_input(img)

img = np.expand_dims(img, axis=0)


In [65]:
prediction = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step


In [66]:
for class_name, probability in zip(classes, prediction[0]):
    print(f"{class_name}: {probability*100:.2f}%")

Acne: 8.61%
Eczemaa: 89.93%
Hyperpigmentation: 0.10%
Normal: 0.02%
Rosacea: 1.32%
Wrinkle: 0.02%


In [67]:
import numpy as np

y_true = []

for images, labels in test_data:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

In [69]:
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)

37/37 ━━━━━━━━━━━━━━━━━━━━ 17s 258ms/step


In [70]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

print(cm)

[[112  90   9   1   6   4]
 [  2 235   6   0   3   0]
 [  7  25  52  27   4   2]
 [  2  12  22 295   2  13]
 [  6  56   0   0  83   0]
 [  1   2   2  10   2  83]]


In [74]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=classes
    )
)

                   precision    recall  f1-score   support

             Acne       0.86      0.50      0.64       222
          Eczemaa       0.56      0.96      0.71       246
Hyperpigmentation       0.57      0.44      0.50       117
           Normal       0.89      0.85      0.87       346
          Rosacea       0.83      0.57      0.68       145
          Wrinkle       0.81      0.83      0.82       100

         accuracy                           0.73      1176
        macro avg       0.75      0.69      0.70      1176
     weighted avg       0.77      0.73      0.73      1176

